# GitHub Actions CI/CD

Automatizar pipelines de test, build y deploy

## Introducción

GitHub Actions permite automatizar tu pipeline de CI/CD directamente desde GitHub. Cada evento en el repositorio (push, pull request, merge) puede disparar workflows que corren tests, construyen imágenes y despliegan a producción.

### Objetivos de Aprendizaje

- Entender la estructura de un workflow de GitHub Actions
- Crear un pipeline de CI que corre tests automáticamente
- Configurar CD para desplegar a un VPS con SSH
- Usar secrets de GitHub para almacenar credenciales de forma segura
- Implementar matrix builds para testear en múltiples versiones de Python

## Estructura de un Workflow

> Un workflow se define en .github/workflows/*.yml. Tiene: name (nombre visible), on (eventos que lo disparan), jobs (conjunto de runners), y cada job tiene steps (pasos individuales).

In [ ]:
workflow_yaml = """
name: CI/CD Pipeline

on:
  push:
    branches: [main, staging]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run tests
        run: pytest tests/

  deploy:
    needs: test
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'
    steps:
      - name: Deploy to server
        run: echo 'Desplegando...'
"""

print("=== Estructura de un Workflow ===")
print(workflow_yaml)

estructura = {
    "name": "Nombre del workflow visible en GitHub",
    "on": "Eventos que disparan el workflow",
    "jobs": "Conjunto de jobs que corren en runners",
    "steps": "Pasos individuales dentro de cada job",
    "uses": "Acciones pre-hechas de la comunidad",
    "run": "Comandos shell personalizados",
}

print("\nComponentes principales:")
for key, val in estructura.items():
    print(f"  {key}: {val}")

## Pipeline de CI: Tests Automáticos

> Un pipeline de CI corre tests en cada push y PR. Esto garantiza que el código nuevo no rompe nada. El job de test es condición necesaria para que el job de deploy pueda correr.

In [ ]:
ci_workflow = """
name: Continuous Integration

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ['3.10', '3.11', '3.12']

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}

      - name: Install dependencies
        run: |
          pip install uv
          uv pip install --system -r requirements.txt

      - name: Lint with ruff
        run: ruff check .

      - name: Type check with mypy
        run: mypy app/

      - name: Run tests
        run: pytest tests/ -v

      - name: Upload coverage
        uses: codecov/codecov-action@v4
        with:
          files: ./coverage.xml
"""

print("=== Pipeline de CI ===")
print(ci_workflow)

print("\nQué pasa en cada evento:")
eventos = {
    "push a main": "Se dispara el workflow completo (test + deploy)",
    "PR a main": "Se dispara solo el job de test",
    "push a feature": "Se dispara solo test (sin deploy)",
}
for evento, accion in eventos.items():
    print(f"  {evento}: {accion}")

## CD: Despliegue a VPS con SSH

> El deployment se dispara solo cuando el push es a main Y todos los tests pasaron. Usa secrets de GitHub para las credenciales SSH — nunca las escribas en el YAML.

In [ ]:
cd_workflow = """
name: Deploy to Production

on:
  push:
    branches: [main]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0

      - name: Deploy to server
        uses: appleboy/ssh-action@v0.1.10
        with:
          host: ${{ secrets.SERVER_HOST }}
          username: ${{ secrets.SERVER_USER }}
          key: ${{ secrets.SERVER_SSH_KEY }}
          script: |
            cd /var/www/mi-app
            git pull origin main
            docker compose down
            docker compose up -d --build
            docker image prune -f
"""

print("=== Workflow de CD ===")
print(cd_workflow)

print("\nSecrets configurados en GitHub:")
secrets_needed = {
    "SERVER_HOST": "IP o dominio del servidor",
    "SERVER_USER": "Usuario SSH (ej: deploy)",
    "SERVER_SSH_KEY": "Clave privada SSH (content, not path)",
}
for name, desc in secrets_needed.items():
    print(f"  {name}: {desc}")
print("\nConfigurar en: Settings > Secrets and variables > Actions")

## Secrets de GitHub para Credenciales Seguras

> Los secrets de GitHub están cifrados y accesibles solo en workflows. Nunca pongas credenciales en texto plano en el YAML. Configura secrets en Settings > Secrets and variables > Actions.

In [ ]:
print("=== Secrets de GitHub ===")

secrets_info = {
    "Dónde residen": "Settings > Secrets and variables > Actions > New repository secret",
    "Acceso": "Solo en workflows, nunca visibles en logs",
    "Nomenclatura": "Usa mayúsculas y guiones bajos: DATABASE_URL",
    "Límite": "500 secrets por repositorio, 64KB por secret",
}

for key, val in secrets_info.items():
    print(f"  {key}: {val}")

print("\nEjemplo de uso en workflow:")
print("  env:")
print("    DATABASE_URL: ${{ secrets.DATABASE_URL }}")
print("  run:")
print("    echo \"Connecting to ${{ secrets.DB_HOST }}\"")

print("\nMétodos de configuración:")
metodos = {
    "UI de GitHub": "Settings > Secrets (manual)",
    "CLI": "gh secret set API_KEY --body 'valor'",
    "Org-level": "Shared across all repos in organization",
}
for metodo, desc in metodos.items():
    print(f"  {metodo}: {desc}")

## Matrix Builds

> La estrategia matrix permite correr el mismo job con múltiples combinaciones de valores. Ejemplo: testear en Python 3.10, 3.11 y 3.12 simultáneamente.

In [ ]:
matrix_yaml = """
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      fail-fast: false
      matrix:
        python-version: ['3.10', '3.11', '3.12']
        os: [ubuntu-latest, windows-latest]

    steps:
      - uses: actions/checkout@v4
      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - name: Run tests
        run: pytest tests/
"""

print("=== Matrix Build ===")
print(matrix_yaml)

print("\nCombinaciones resultantes (3 x 2 = 6 jobs):")
combinations = [
    ("3.10", "ubuntu"), ("3.10", "windows"),
    ("3.11", "ubuntu"), ("3.11", "windows"),
    ("3.12", "ubuntu"), ("3.12", "windows"),
]
for py, os in combinations:
    print(f"  Python {py} on {os}")

print("\nfail-fast: false significa que si uno falla, los demás continúan")

## Workflow Completo de CI/CD

Workflow completo que combina CI (tests) y CD (despliegue) con las mejores prácticas.

In [ ]:
complete_workflow = """
name: CI/CD Pipeline

on:
  push:
    branches: [main, staging]
  pull_request:
    branches: [main]

env:
  PYTHON_VERSION: '3.12'

jobs:
  # ── CI: Tests ──────────────────────────────────────────────
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Lint
        run: ruff check . && ruff format --check .
      - name: Type check
        run: mypy app/
      - name: Test
        run: pytest tests/ -v --cov=app --cov-report=xml
      - name: Upload coverage
        uses: codecov/codecov-action@v4

  # ── CD: Deploy to Staging ──────────────────────────────────
  deploy-staging:
    runs-on: ubuntu-latest
    needs: test
    if: github.ref == 'refs/heads/staging'
    environment: staging
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to staging
        uses: appleboy/ssh-action@v0.1.10
        with:
          host: ${{ secrets.STAGING_HOST }}
          username: ${{ secrets.STAGING_USER }}
          key: ${{ secrets.STAGING_SSH_KEY }}
          script: |
            cd /var/www/staging && git pull && docker compose up -d --build

  # ── CD: Deploy to Production ──────────────────────────────
  deploy-production:
    runs-on: ubuntu-latest
    needs: test
    if: github.ref == 'refs/heads/main'
    environment: production
    steps:
      - uses: actions/checkout@v4
      - name: Deploy to production
        uses: appleboy/ssh-action@v0.1.10
        with:
          host: ${{ secrets.PROD_HOST }}
          username: ${{ secrets.PROD_USER }}
          key: ${{ secrets.PROD_SSH_KEY }}
          script: |
            cd /var/www/production && git pull && docker compose up -d --build
"""

print("=== Workflow Completo ===")
print(complete_workflow)

print("\nFlujo de trabajo:")
flujo = [
    ("Push a staging", "Test → Deploy staging"),
    ("Push a main", "Test → Deploy production"),
    ("PR a main", "Solo test (no deploy)"),
]
for trigger, result in flujo:
    print(f"  {trigger}: {result}")

## Tips y Mejores Prácticas

> Usa needs: test para que deploy solo corra si todos los tests pasaron. Esto evita desplegar código roto.

> Configure environments (Settings > Environments) para añadir protection rules y required reviewers para producción.

> Usa actions/checkout@v4 con fetch-depth: 0 si necesitas el historial completo de git en el runner.

> Agrupa logs relacionados: usa ::group:: para collapsar logs de un mismo paso en la UI de GitHub.

> Mantén los secrets en mayúsculas con guiones bajos: DATABASE_URL, API_KEY_SECRET.

> Para speeds, usa pip cache: actions/setup-python@v5 con cache: 'pip'.

## Errores Comunes

### Credenciales en texto plano en el YAML

¿Por qué ocurre?
- Escribir password: 'mi-password-secreto' directamente en el workflow.

Solución
- Usa ${{ secrets.MY_SECRET }} para referenciar secrets cifrados. Configura en Settings > Secrets.

### Deploy sin jobs de test

¿Por qué ocurre?
- No usar needs: test o no tener condición if para verificar que los tests pasaron.

Solución
- Añade needs: test al job de deploy y if: github.ref == 'refs/heads/main' para controlar cuando desplegar.

### Secrets no disponibles en pull_request desde forks

¿Por qué ocurre?
- Por seguridad, GitHub no expone secrets a workflows disparados por forks.

Solución
- Los tests de PRs de forks solo pueden usar secrets si son de colaboradores con acceso. Usa jobs separados: test público (sin secrets) y deploy privado.

### Usar secrets en logs inadvertently

¿Por qué ocurre?
- echo ${{ secrets.API_KEY }} imprime el valor completo en los logs de Actions.

Solución
- Nunca imprimas secrets directamente. Si necesitas debugging, usa máscaras: ::add-mask::valor

### No usar environment protection rules

¿Por qué ocurre?
- Cualquier push a main despliega a producción sin aprobación humana.

Solución
- Configura environments con required reviewers en Settings > Environments > production.